# Очистка EDF-записей от артефактов

Этот ноутбук:
- загружает все `.edf` файлы из текущей папки;
- делит каждую запись на эпохи по 10 секунд;
- находит эпохи с артефактами по выбросам амплитуды;
- удаляет плохие эпохи;
- сохраняет очищенные записи в папку `cleaned_data`;
- сохраняет таблицу с информацией, какие эпохи были вырезаны для каждого файла.


In [10]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import mne


In [11]:
epoch_duration = 10
threshold_factor = 2.0

input_dir = Path(".")
output_dir = Path("cleaned_data")
output_dir.mkdir(exist_ok=True)

edf_files = sorted(input_dir.glob("*.edf"))
[file.name for file in edf_files]


['1 open eyes.edf',
 '2 closed eyes.edf',
 '3 2026.01.24-16.16.21.085.edf',
 '4 2026.01.24-16.18.47.499.edf',
 '5 2026.01.24-16.38.37.757.edf',
 '6 2026.01.24-16.41.01.794.edf',
 '7 2026.01.24-16.53.06.041.edf',
 '8 2026.01.24-17.03.42.427.edf',
 '9 2026.01.24-17.05.59.615.edf']

In [12]:
def detect_bad_epochs(raw, epoch_duration=10, threshold_factor=2.0):
    data = raw.get_data()
    sfreq = raw.info["sfreq"]

    samples_per_epoch = int(epoch_duration * sfreq)
    n_channels, n_samples = data.shape
    n_epochs = n_samples // samples_per_epoch

    trimmed_samples = n_epochs * samples_per_epoch
    data_trimmed = data[:, :trimmed_samples]

    epoch_max_matrix = np.zeros((n_epochs, n_channels))

    for epoch_idx in range(n_epochs):
        start = epoch_idx * samples_per_epoch
        end = start + samples_per_epoch
        epoch = data_trimmed[:, start:end]
        epoch_max_matrix[epoch_idx] = np.max(np.abs(epoch), axis=1)

    channel_typical_max = np.median(epoch_max_matrix, axis=0)

    good_epoch_indices = []
    bad_epoch_indices = []

    for epoch_idx in range(n_epochs):
        epoch_max_amp = epoch_max_matrix[epoch_idx]
        is_bad = np.any(epoch_max_amp > threshold_factor * channel_typical_max)

        if is_bad:
            bad_epoch_indices.append(epoch_idx)
        else:
            good_epoch_indices.append(epoch_idx)

    return {
        "data_trimmed": data_trimmed,
        "sfreq": sfreq,
        "samples_per_epoch": samples_per_epoch,
        "n_epochs": n_epochs,
        "good_epoch_indices": good_epoch_indices,
        "bad_epoch_indices": bad_epoch_indices,
        "channel_typical_max": channel_typical_max,
        "epoch_max_matrix": epoch_max_matrix,
    }


In [13]:
def build_clean_data(data_trimmed, good_epoch_indices, samples_per_epoch):
    if not good_epoch_indices:
        return None

    good_segments = []

    for epoch_idx in good_epoch_indices:
        start = epoch_idx * samples_per_epoch
        end = start + samples_per_epoch
        good_segments.append(data_trimmed[:, start:end])

    return np.concatenate(good_segments, axis=1)


In [14]:
summary_rows = []

for edf_path in edf_files:
    print(f"Processing: {edf_path.name}")

    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose="ERROR")

    result = detect_bad_epochs(
        raw,
        epoch_duration=epoch_duration,
        threshold_factor=threshold_factor,
    )

    clean_data = build_clean_data(
        result["data_trimmed"],
        result["good_epoch_indices"],
        result["samples_per_epoch"],
    )

    if clean_data is None:
        print(f"  No clean epochs left, skipping save: {edf_path.name}")
        summary_rows.append(
            {
                "file_name": edf_path.name,
                "n_epochs": result["n_epochs"],
                "good_epochs": 0,
                "bad_epochs": len(result["bad_epoch_indices"]),
                "removed_epoch_indices": result["bad_epoch_indices"],
                "original_duration_sec": result["data_trimmed"].shape[1] / result["sfreq"],
                "clean_duration_sec": 0,
                "saved_file": None,
            }
        )
        continue

    raw_clean = mne.io.RawArray(clean_data, raw.info.copy(), verbose="ERROR")

    output_file = output_dir / f"{edf_path.stem}_clean.edf"
    raw_clean.export(output_file, fmt="edf")

    summary_rows.append(
        {
            "file_name": edf_path.name,
            "n_epochs": result["n_epochs"],
            "good_epochs": len(result["good_epoch_indices"]),
            "bad_epochs": len(result["bad_epoch_indices"]),
            "removed_epoch_indices": result["bad_epoch_indices"],
            "original_duration_sec": result["data_trimmed"].shape[1] / result["sfreq"],
            "clean_duration_sec": clean_data.shape[1] / result["sfreq"],
            "saved_file": str(output_file),
        }
    )

    print(f"  Removed epochs: {result['bad_epoch_indices']}")
    print(f"  Saved: {output_file.name}")


Processing: 1 open eyes.edf
  Removed epochs: [0, 3]
  Saved: 1 open eyes_clean.edf
Processing: 2 closed eyes.edf
  Removed epochs: [5]
  Saved: 2 closed eyes_clean.edf
Processing: 3 2026.01.24-16.16.21.085.edf
  Removed epochs: [0, 8]
  Saved: 3 2026.01.24-16.16.21.085_clean.edf
Processing: 4 2026.01.24-16.18.47.499.edf
  Removed epochs: [5, 10]
  Saved: 4 2026.01.24-16.18.47.499_clean.edf
Processing: 5 2026.01.24-16.38.37.757.edf
  Removed epochs: [0, 5, 9, 10, 11]
  Saved: 5 2026.01.24-16.38.37.757_clean.edf
Processing: 6 2026.01.24-16.41.01.794.edf
  Removed epochs: [0, 1, 2, 3, 4, 5, 6, 10, 11, 15]
  Saved: 6 2026.01.24-16.41.01.794_clean.edf
Processing: 7 2026.01.24-16.53.06.041.edf
  Removed epochs: []
  Saved: 7 2026.01.24-16.53.06.041_clean.edf
Processing: 8 2026.01.24-17.03.42.427.edf
  Removed epochs: [7, 11]
  Saved: 8 2026.01.24-17.03.42.427_clean.edf
Processing: 9 2026.01.24-17.05.59.615.edf
  Removed epochs: []
  Saved: 9 2026.01.24-17.05.59.615_clean.edf


In [15]:
summary_df = pd.DataFrame(summary_rows)
summary_df


,file_name,n_epochs,good_epochs,bad_epochs,removed_epoch_indices,original_duration_sec,clean_duration_sec,saved_file
0,1 open eyes.edf,6,4,2,"[0, 3]",60.0,40.0,cleaned_data/1 open eyes_clean.edf
1,2 closed eyes.edf,6,5,1,[5],60.0,50.0,cleaned_data/2 closed eyes_clean.edf
2,3 2026.01.24-16.16.21.085.edf,12,10,2,"[0, 8]",120.0,100.0,cleaned_data/3 2026.01.24-16.16.21.085_clean.edf
3,4 2026.01.24-16.18.47.499.edf,12,10,2,"[5, 10]",120.0,100.0,cleaned_data/4 2026.01.24-16.18.47.499_clean.edf
4,5 2026.01.24-16.38.37.757.edf,12,7,5,"[0, 5, 9, 10, 11]",120.0,70.0,cleaned_data/5 2026.01.24-16.38.37.757_clean.edf
5,6 2026.01.24-16.41.01.794.edf,18,8,10,"[0, 1, 2, 3, 4, 5, 6, 10, 11, 15]",180.0,80.0,cleaned_data/6 2026.01.24-16.41.01.794_clean.edf
6,7 2026.01.24-16.53.06.041.edf,1,1,0,[],10.0,10.0,cleaned_data/7 2026.01.24-16.53.06.041_clean.edf
7,8 2026.01.24-17.03.42.427.edf,12,10,2,"[7, 11]",120.0,100.0,cleaned_data/8 2026.01.24-17.03.42.427_clean.edf
8,9 2026.01.24-17.05.59.615.edf,46,46,0,[],460.0,460.0,cleaned_data/9 2026.01.24-17.05.59.615_clean.edf


In [16]:
summary_df.to_csv(output_dir / "cleaning_summary.csv", index=False)

with open(output_dir / "cleaning_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_rows, f, ensure_ascii=False, indent=2)


In [17]:
summary_df[["file_name", "n_epochs", "good_epochs", "bad_epochs", "removed_epoch_indices", "clean_duration_sec"]]


,file_name,n_epochs,good_epochs,bad_epochs,removed_epoch_indices,clean_duration_sec
0,1 open eyes.edf,6,4,2,"[0, 3]",40.0
1,2 closed eyes.edf,6,5,1,[5],50.0
2,3 2026.01.24-16.16.21.085.edf,12,10,2,"[0, 8]",100.0
3,4 2026.01.24-16.18.47.499.edf,12,10,2,"[5, 10]",100.0
4,5 2026.01.24-16.38.37.757.edf,12,7,5,"[0, 5, 9, 10, 11]",70.0
5,6 2026.01.24-16.41.01.794.edf,18,8,10,"[0, 1, 2, 3, 4, 5, 6, 10, 11, 15]",80.0
6,7 2026.01.24-16.53.06.041.edf,1,1,0,[],10.0
7,8 2026.01.24-17.03.42.427.edf,12,10,2,"[7, 11]",100.0
8,9 2026.01.24-17.05.59.615.edf,46,46,0,[],460.0
